# 04 · Score inteligente de salud financiera V2

Esta versión corrige el comportamiento de la versión anterior: ahora genera score para **todos los años disponibles** y no solo para el último año.

## Criterio
- Años históricos: score con `RandomForestRegressor` + `KMeans`.
- Último año disponible: score con `RandomForestRegressor` + `KMeans` + `Prophet`.

Esto evita aplicar proyecciones futuras a años pasados.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

## 1. Rutas del proyecto

In [2]:
def encontrar_raiz_proyecto() -> Path:
    actual = Path.cwd().resolve()
    for ruta in [actual] + list(actual.parents):
        if (ruta / "01_datos_procesados").exists():
            return ruta
    raise FileNotFoundError("No se encontró 01_datos_procesados. Ejecuta este notebook dentro del proyecto GRUPO_02.")

RAIZ_PROYECTO = encontrar_raiz_proyecto()
CARPETA_MODELOS_RESULTADOS = RAIZ_PROYECTO / "01_datos_procesados" / "modelos"
CARPETA_MODELOS_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Raíz:", RAIZ_PROYECTO)
print("Modelos/resultados:", CARPETA_MODELOS_RESULTADOS)

Raíz: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador
Modelos/resultados: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos


## 2. Funciones auxiliares

In [3]:
def cargar_parquet_o_csv(nombre_base: str, carpeta: Path) -> pd.DataFrame:
    ruta_parquet = carpeta / f"{nombre_base}.parquet"
    ruta_csv = carpeta / f"{nombre_base}.csv"
    if ruta_parquet.exists():
        print("Leyendo:", ruta_parquet)
        return pd.read_parquet(ruta_parquet)
    if ruta_csv.exists():
        print("Leyendo:", ruta_csv)
        return pd.read_csv(ruta_csv)
    print(f"No se encontró {nombre_base}")
    return pd.DataFrame()

def exportar_csv_parquet(df: pd.DataFrame, nombre_base: str):
    ruta_csv = CARPETA_MODELOS_RESULTADOS / f"{nombre_base}.csv"
    ruta_parquet = CARPETA_MODELOS_RESULTADOS / f"{nombre_base}.parquet"
    df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
    try:
        df.to_parquet(ruta_parquet, index=False)
        print("Exportado:", ruta_csv)
        print("Exportado:", ruta_parquet)
    except Exception as e:
        print("Exportado CSV:", ruta_csv)
        print("No se pudo exportar Parquet:", repr(e))

def limpiar_banco(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.upper()

## 3. Cargar resultados de modelos

In [4]:
df_roe = cargar_parquet_o_csv("modelo_roe_predicciones", CARPETA_MODELOS_RESULTADOS)
df_clusters = cargar_parquet_o_csv("clusters_bancos_anual", CARPETA_MODELOS_RESULTADOS)
df_prophet = cargar_parquet_o_csv("prophet_forecast", CARPETA_MODELOS_RESULTADOS)

print("ROE:", df_roe.shape)
print("Clusters:", df_clusters.shape)
print("Prophet:", df_prophet.shape)

Leyendo: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\modelo_roe_predicciones.parquet
Leyendo: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\clusters_bancos_anual.parquet
Leyendo: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\prophet_forecast.parquet
ROE: (4931, 15)
Clusters: (441, 12)
Prophet: (21056, 12)


## 4. Componente ROE real vs estimado por banco y año

In [5]:
if df_roe.empty:
    df_roe_componente = pd.DataFrame()
else:
    df_roe = df_roe.copy()
    df_roe["banco_estandarizado"] = limpiar_banco(df_roe["banco_estandarizado"])
    df_roe["anio"] = pd.to_numeric(df_roe["anio"], errors="coerce")
    if "error_roe" not in df_roe.columns:
        df_roe["error_roe"] = df_roe["roe_real"] - df_roe["roe_estimado"]
    if "error_abs_roe" not in df_roe.columns:
        df_roe["error_abs_roe"] = df_roe["error_roe"].abs()

    df_roe_componente = (
        df_roe.groupby(["anio", "banco_estandarizado"], dropna=False)
        .agg(
            roe_real=("roe_real", "mean"),
            roe_estimado=("roe_estimado", "mean"),
            error_roe=("error_roe", "mean"),
            error_abs_roe=("error_abs_roe", "mean")
        )
        .reset_index()
    )

    # Mayor error positivo significa ROE real superior al esperado por el modelo.
    df_roe_componente["puntaje_roe_modelo"] = (
        df_roe_componente.groupby("anio")["error_roe"].rank(pct=True) * 100
    ).round(2)

display(df_roe_componente.head())

,anio,banco_estandarizado,roe_real,roe_estimado,error_roe,error_abs_roe,puntaje_roe_modelo
0,2009,AMAZONAS,8.990000,9.883329,-0.893329,1.556620,34.62
1,2009,AUSTRO,20.646667,19.433323,1.213344,1.953336,80.77
2,2009,BOLIVARIANO,13.958333,13.739272,0.219062,0.419903,65.38
3,2009,CAPITAL,10.010833,12.177361,-2.166528,2.318175,7.69
4,2009,CITIBANK,13.300000,14.572863,-1.272863,2.560658,26.92


## 5. Componente KMeans por banco y año

In [6]:
if df_clusters.empty:
    df_cluster_componente = pd.DataFrame()
else:
    df_cluster_componente = df_clusters.copy()
    df_cluster_componente["banco_estandarizado"] = limpiar_banco(df_cluster_componente["banco_estandarizado"])
    df_cluster_componente["anio"] = pd.to_numeric(df_cluster_componente["anio"], errors="coerce")

    if "perfil_cluster" not in df_cluster_componente.columns:
        df_cluster_componente["perfil_cluster"] = "Perfil intermedio"

    mapa_puntaje_cluster = {
        "Bancos grandes o consolidados": 85,
        "Pequeños con mayor solvencia proxy": 75,
        "Perfil intermedio": 55,
        "Mayor riesgo relativo": 30
    }

    df_cluster_componente["puntaje_cluster"] = (
        df_cluster_componente["perfil_cluster"].map(mapa_puntaje_cluster).fillna(55)
    )

display(df_cluster_componente.head())

,anio,banco_estandarizado,meses_con_dato,activos_totales,patrimonio,roe,morosidad,solvencia_proxy,cluster,pca_1,pca_2,perfil_cluster,puntaje_cluster
0,2009,AMAZONAS,12,120.492500,11.405833,8.990000,3.854167,7.836667,2,-0.475294,-0.480742,Mayor riesgo relativo,30
1,2009,AUSTRO,12,647.960833,57.102500,20.646667,7.670000,7.409167,2,-0.223535,-0.793598,Mayor riesgo relativo,30
2,2009,BOLIVARIANO,12,1271.975833,109.273333,13.958333,1.559167,7.487500,2,0.071318,-0.718681,Mayor riesgo relativo,30
3,2009,CAPITAL,12,62.264167,11.842500,10.010833,10.998333,16.355000,2,-0.916158,0.123199,Mayor riesgo relativo,30
4,2009,CITIBANK,12,302.596667,29.135833,13.300000,16.335833,8.315000,2,-0.538688,-0.005709,Mayor riesgo relativo,30


## 6. Componentes Prophet solo para el último año

In [7]:
def obtener_base_y_proyeccion(grupo: pd.DataFrame) -> pd.Series:
    grupo = grupo.sort_values("ds").copy()
    historico = grupo[(grupo["valor_real"].notna()) | (grupo.get("tipo", "") == "historico_ajustado")]
    proyeccion = grupo[grupo.get("tipo", "") == "proyeccion"]
    if historico.empty or proyeccion.empty:
        return pd.Series({"valor_base": np.nan, "valor_proyectado": np.nan, "cambio_proyectado": np.nan, "cambio_proyectado_pct": np.nan})

    ultima_hist = historico.iloc[-1]
    valor_base = ultima_hist["valor_real"] if pd.notna(ultima_hist.get("valor_real")) else ultima_hist["yhat"]
    valor_proyectado = proyeccion["yhat"].iloc[-1]
    cambio = valor_proyectado - valor_base
    cambio_pct = (cambio / abs(valor_base)) * 100 if pd.notna(valor_base) and valor_base != 0 else np.nan

    return pd.Series({"valor_base": valor_base, "valor_proyectado": valor_proyectado, "cambio_proyectado": cambio, "cambio_proyectado_pct": cambio_pct})

if df_prophet.empty:
    df_prophet_scores = pd.DataFrame()
else:
    df_prophet_tmp = df_prophet.copy()
    df_prophet_tmp["banco_estandarizado"] = limpiar_banco(df_prophet_tmp["banco_estandarizado"])
    df_prophet_tmp["indicador"] = df_prophet_tmp["indicador"].astype(str).str.strip().str.lower()
    df_prophet_tmp["ds"] = pd.to_datetime(df_prophet_tmp["ds"], errors="coerce")

    df_prophet_componentes = (
        df_prophet_tmp.groupby(["banco_estandarizado", "indicador"], dropna=False)
        .apply(obtener_base_y_proyeccion)
        .reset_index()
    )

    df_prophet_componentes = df_prophet_componentes.pivot_table(
        index="banco_estandarizado",
        columns="indicador",
        values=["valor_base", "valor_proyectado", "cambio_proyectado", "cambio_proyectado_pct"],
        aggfunc="first"
    )
    df_prophet_componentes.columns = [f"{metrica}_{indicador}" for metrica, indicador in df_prophet_componentes.columns]
    df_prophet_scores = df_prophet_componentes.reset_index()

display(df_prophet_scores.head())

,banco_estandarizado,cambio_proyectado_activos_totales,cambio_proyectado_morosidad,cambio_proyectado_roe,cambio_proyectado_solvencia_proxy,cambio_proyectado_pct_activos_totales,cambio_proyectado_pct_morosidad,cambio_proyectado_pct_roe,cambio_proyectado_pct_solvencia_proxy,valor_base_activos_totales,valor_base_morosidad,valor_base_roe,valor_base_solvencia_proxy,valor_proyectado_activos_totales,valor_proyectado_morosidad,valor_proyectado_roe,valor_proyectado_solvencia_proxy
0,AMAZONAS,23.400951,0.602184,21.083597,-0.654262,5.103250,19.362825,94.928397,-8.260890,458.55,3.11,22.21,7.92,481.950951,3.712184,43.293597,7.265738
1,AUSTRO,57.574993,-1.064041,-4.792276,-0.142174,1.631912,-31.387651,-47.589631,-2.081603,3528.07,3.39,10.07,6.83,3585.644993,2.325959,5.277724,6.687826
2,BOLIVARIANO,238.324825,15.200429,-0.141018,0.328003,3.793320,938.298083,-1.281979,3.664834,6282.75,1.62,11.00,8.95,6521.074825,16.820429,10.858982,9.278003
3,CAPITAL,-57.233173,-7.831432,-20.944103,-1.871423,-69.163955,-248.616888,-512.080758,-18.275620,82.75,3.15,-4.09,10.24,25.516827,-4.681432,-25.034103,8.368577
4,CITIBANK,-71.938371,-2.404970,-2.259327,0.150778,-8.120371,NaN,-12.114352,1.261742,885.90,0.00,18.65,11.95,813.961629,-2.404970,16.390673,12.100778


## 7. Convertir Prophet a puntajes

In [8]:
def score_cambio_pct(cambio_pct, mayor_es_mejor=True, limite=20):
    if pd.isna(cambio_pct):
        return np.nan
    cambio_pct = max(-limite, min(limite, cambio_pct))
    score = 50 + (cambio_pct / limite) * 50 if mayor_es_mejor else 50 - (cambio_pct / limite) * 50
    return round(max(0, min(100, score)), 2)

def score_cambio_puntos(cambio_puntos, mayor_es_mejor=True, limite=10):
    if pd.isna(cambio_puntos):
        return np.nan
    cambio_puntos = max(-limite, min(limite, cambio_puntos))
    score = 50 + (cambio_puntos / limite) * 50 if mayor_es_mejor else 50 - (cambio_puntos / limite) * 50
    return round(max(0, min(100, score)), 2)

if not df_prophet_scores.empty:
    df_prophet_scores["puntaje_activos_proyectados"] = df_prophet_scores.get("cambio_proyectado_pct_activos_totales", np.nan).apply(lambda x: score_cambio_pct(x, True, 20))
    df_prophet_scores["puntaje_roe_proyectado"] = df_prophet_scores.get("cambio_proyectado_roe", np.nan).apply(lambda x: score_cambio_puntos(x, True, 10))
    df_prophet_scores["puntaje_morosidad_proyectada"] = df_prophet_scores.get("cambio_proyectado_morosidad", np.nan).apply(lambda x: score_cambio_puntos(x, False, 5))
    df_prophet_scores["puntaje_solvencia_proyectada"] = df_prophet_scores.get("cambio_proyectado_solvencia_proxy", np.nan).apply(lambda x: score_cambio_puntos(x, True, 10))
    df_prophet_scores["puntaje_tendencia_general"] = df_prophet_scores[["puntaje_activos_proyectados", "puntaje_roe_proyectado"]].mean(axis=1, skipna=True)

display(df_prophet_scores.head())

,banco_estandarizado,cambio_proyectado_activos_totales,cambio_proyectado_morosidad,cambio_proyectado_roe,cambio_proyectado_solvencia_proxy,cambio_proyectado_pct_activos_totales,cambio_proyectado_pct_morosidad,cambio_proyectado_pct_roe,cambio_proyectado_pct_solvencia_proxy,valor_base_activos_totales,valor_base_morosidad,valor_base_roe,valor_base_solvencia_proxy,valor_proyectado_activos_totales,valor_proyectado_morosidad,valor_proyectado_roe,valor_proyectado_solvencia_proxy,puntaje_activos_proyectados,puntaje_roe_proyectado,puntaje_morosidad_proyectada,puntaje_solvencia_proyectada,puntaje_tendencia_general
0,AMAZONAS,23.400951,0.602184,21.083597,-0.654262,5.103250,19.362825,94.928397,-8.260890,458.55,3.11,22.21,7.92,481.950951,3.712184,43.293597,7.265738,62.76,100.00,43.98,46.73,81.380
1,AUSTRO,57.574993,-1.064041,-4.792276,-0.142174,1.631912,-31.387651,-47.589631,-2.081603,3528.07,3.39,10.07,6.83,3585.644993,2.325959,5.277724,6.687826,54.08,26.04,60.64,49.29,40.060
2,BOLIVARIANO,238.324825,15.200429,-0.141018,0.328003,3.793320,938.298083,-1.281979,3.664834,6282.75,1.62,11.00,8.95,6521.074825,16.820429,10.858982,9.278003,59.48,49.29,0.00,51.64,54.385
3,CAPITAL,-57.233173,-7.831432,-20.944103,-1.871423,-69.163955,-248.616888,-512.080758,-18.275620,82.75,3.15,-4.09,10.24,25.516827,-4.681432,-25.034103,8.368577,0.00,0.00,100.00,40.64,0.000
4,CITIBANK,-71.938371,-2.404970,-2.259327,0.150778,-8.120371,NaN,-12.114352,1.261742,885.90,0.00,18.65,11.95,813.961629,-2.404970,16.390673,12.100778,29.70,38.70,74.05,50.75,34.200


## 8. Calcular score para todos los años

In [9]:
bases = []

if not df_cluster_componente.empty:
    bases.append(df_cluster_componente[["anio", "banco_estandarizado"]])

if not df_roe_componente.empty:
    bases.append(df_roe_componente[["anio", "banco_estandarizado"]])

if not bases:
    raise ValueError("No existen componentes para construir el score.")

df_score = pd.concat(bases, ignore_index=True).drop_duplicates()
ANIO_SCORE_ACTUAL = int(df_score["anio"].max())
print("Último año con score actual:", ANIO_SCORE_ACTUAL)

if not df_cluster_componente.empty:
    df_score = df_score.merge(df_cluster_componente, on=["anio", "banco_estandarizado"], how="left")

if not df_roe_componente.empty:
    df_score = df_score.merge(df_roe_componente, on=["anio", "banco_estandarizado"], how="left", suffixes=("", "_roe_modelo"))

if not df_prophet_scores.empty:
    df_score = df_score.merge(df_prophet_scores, on="banco_estandarizado", how="left")

    columnas_prophet = [
        "puntaje_tendencia_general", "puntaje_morosidad_proyectada", "puntaje_solvencia_proyectada",
        "puntaje_activos_proyectados", "puntaje_roe_proyectado"
    ]

    # Prophet solo se aplica al último año. Para históricos queda NaN.
    for col in columnas_prophet:
        if col in df_score.columns:
            df_score.loc[df_score["anio"] != ANIO_SCORE_ACTUAL, col] = np.nan

PESOS_SCORE = {
    "puntaje_roe_modelo": 0.25,
    "puntaje_cluster": 0.20,
    "puntaje_tendencia_general": 0.20,
    "puntaje_morosidad_proyectada": 0.20,
    "puntaje_solvencia_proyectada": 0.15
}

for col in PESOS_SCORE:
    if col not in df_score.columns:
        df_score[col] = np.nan

def calcular_score(fila):
    suma = 0
    peso_disp = 0
    for comp, peso in PESOS_SCORE.items():
        valor = fila.get(comp)
        if pd.notna(valor):
            suma += valor * peso
            peso_disp += peso
    return np.nan if peso_disp == 0 else round(suma / peso_disp, 2)

df_score["score_salud"] = df_score.apply(calcular_score, axis=1)
df_score["tipo_score"] = np.where(df_score["anio"] == ANIO_SCORE_ACTUAL, "actual_con_proyecciones", "historico_sin_proyecciones")

display(df_score.head())

Último año con score actual: 2026


,anio,banco_estandarizado,meses_con_dato,activos_totales,patrimonio,roe,morosidad,solvencia_proxy,cluster,pca_1,pca_2,perfil_cluster,puntaje_cluster,roe_real,roe_estimado,error_roe,error_abs_roe,puntaje_roe_modelo,cambio_proyectado_activos_totales,cambio_proyectado_morosidad,cambio_proyectado_roe,cambio_proyectado_solvencia_proxy,cambio_proyectado_pct_activos_totales,cambio_proyectado_pct_morosidad,cambio_proyectado_pct_roe,cambio_proyectado_pct_solvencia_proxy,valor_base_activos_totales,valor_base_morosidad,valor_base_roe,valor_base_solvencia_proxy,valor_proyectado_activos_totales,valor_proyectado_morosidad,valor_proyectado_roe,valor_proyectado_solvencia_proxy,puntaje_activos_proyectados,puntaje_roe_proyectado,puntaje_morosidad_proyectada,puntaje_solvencia_proyectada,puntaje_tendencia_general,score_salud,tipo_score
0,2009,AMAZONAS,12,120.492500,11.405833,8.990000,3.854167,7.836667,2,-0.475294,-0.480742,Mayor riesgo relativo,30,8.990000,9.883329,-0.893329,1.556620,34.62,23.400951,0.602184,21.083597,-0.654262,5.103250,19.362825,94.928397,-8.260890,458.55,3.11,22.21,7.92,481.950951,3.712184,43.293597,7.265738,NaN,NaN,NaN,NaN,NaN,32.57,historico_sin_proyecciones
1,2009,AUSTRO,12,647.960833,57.102500,20.646667,7.670000,7.409167,2,-0.223535,-0.793598,Mayor riesgo relativo,30,20.646667,19.433323,1.213344,1.953336,80.77,57.574993,-1.064041,-4.792276,-0.142174,1.631912,-31.387651,-47.589631,-2.081603,3528.07,3.39,10.07,6.83,3585.644993,2.325959,5.277724,6.687826,NaN,NaN,NaN,NaN,NaN,58.21,historico_sin_proyecciones
2,2009,BOLIVARIANO,12,1271.975833,109.273333,13.958333,1.559167,7.487500,2,0.071318,-0.718681,Mayor riesgo relativo,30,13.958333,13.739272,0.219062,0.419903,65.38,238.324825,15.200429,-0.141018,0.328003,3.793320,938.298083,-1.281979,3.664834,6282.75,1.62,11.00,8.95,6521.074825,16.820429,10.858982,9.278003,NaN,NaN,NaN,NaN,NaN,49.66,historico_sin_proyecciones
3,2009,CAPITAL,12,62.264167,11.842500,10.010833,10.998333,16.355000,2,-0.916158,0.123199,Mayor riesgo relativo,30,10.010833,12.177361,-2.166528,2.318175,7.69,-57.233173,-7.831432,-20.944103,-1.871423,-69.163955,-248.616888,-512.080758,-18.275620,82.75,3.15,-4.09,10.24,25.516827,-4.681432,-25.034103,8.368577,NaN,NaN,NaN,NaN,NaN,17.61,historico_sin_proyecciones
4,2009,CITIBANK,12,302.596667,29.135833,13.300000,16.335833,8.315000,2,-0.538688,-0.005709,Mayor riesgo relativo,30,13.300000,14.572863,-1.272863,2.560658,26.92,-71.938371,-2.404970,-2.259327,0.150778,-8.120371,NaN,-12.114352,1.261742,885.90,0.00,18.65,11.95,813.961629,-2.404970,16.390673,12.100778,NaN,NaN,NaN,NaN,NaN,28.29,historico_sin_proyecciones


## 9. Lectura ejecutiva y ranking por año

In [10]:
def clasificar_lectura(score):
    if pd.isna(score):
        return "Sin información suficiente"
    if score >= 75:
        return "Salud financiera favorable"
    if score >= 60:
        return "Perfil financiero sólido"
    if score >= 45:
        return "Perfil intermedio con seguimiento"
    return "Requiere revisión"

def analisis_ejecutivo(fila):
    banco = fila.get("banco_estandarizado", "El banco")
    anio = int(fila["anio"]) if pd.notna(fila.get("anio")) else "N/D"
    score = fila.get("score_salud", np.nan)
    lectura = fila.get("lectura_score", "Sin información suficiente")
    partes = []

    if pd.notna(score):
        partes.append(f"En {anio}, {banco} obtiene un score de salud financiera de {score:.2f}/100, clasificado como: {lectura}.")
    else:
        partes.append(f"En {anio}, {banco} no cuenta con información suficiente para calcular un score completo.")

    if pd.notna(fila.get("error_roe")):
        if fila["error_roe"] > 0:
            partes.append("El ROE real se ubica por encima del ROE estimado por el modelo.")
        elif fila["error_roe"] < 0:
            partes.append("El ROE real se ubica por debajo del ROE estimado por el modelo.")

    if pd.notna(fila.get("perfil_cluster")):
        partes.append(f"Según KMeans, pertenece al perfil: {fila['perfil_cluster']}.")

    if fila.get("tipo_score") == "actual_con_proyecciones":
        if pd.notna(fila.get("cambio_proyectado_morosidad")):
            cambio = fila["cambio_proyectado_morosidad"]
            if cambio > 0:
                partes.append(f"La morosidad proyectada aumenta en {cambio:.2f} puntos porcentuales.")
            elif cambio < 0:
                partes.append(f"La morosidad proyectada disminuye en {abs(cambio):.2f} puntos porcentuales.")

        if pd.notna(fila.get("cambio_proyectado_solvencia_proxy")):
            cambio = fila["cambio_proyectado_solvencia_proxy"]
            if cambio > 0:
                partes.append(f"La solvencia proxy proyectada mejora en {cambio:.2f} puntos porcentuales.")
            elif cambio < 0:
                partes.append(f"La solvencia proxy proyectada disminuye en {abs(cambio):.2f} puntos porcentuales.")
    else:
        partes.append("Este score histórico no incorpora Prophet para evitar usar proyecciones futuras en años pasados.")

    return " ".join(partes)

df_score["lectura_score"] = df_score["score_salud"].apply(clasificar_lectura)
df_score = df_score.sort_values(["anio", "score_salud"], ascending=[True, False]).reset_index(drop=True)
df_score["ranking_score"] = df_score.groupby("anio")["score_salud"].rank(method="first", ascending=False).astype("Int64")
df_score["analisis_ejecutivo"] = df_score.apply(analisis_ejecutivo, axis=1)

display(df_score.head(20))

,anio,banco_estandarizado,meses_con_dato,activos_totales,patrimonio,roe,morosidad,solvencia_proxy,cluster,pca_1,pca_2,perfil_cluster,puntaje_cluster,roe_real,roe_estimado,error_roe,error_abs_roe,puntaje_roe_modelo,cambio_proyectado_activos_totales,cambio_proyectado_morosidad,cambio_proyectado_roe,cambio_proyectado_solvencia_proxy,cambio_proyectado_pct_activos_totales,cambio_proyectado_pct_morosidad,cambio_proyectado_pct_roe,cambio_proyectado_pct_solvencia_proxy,valor_base_activos_totales,valor_base_morosidad,valor_base_roe,valor_base_solvencia_proxy,valor_proyectado_activos_totales,valor_proyectado_morosidad,valor_proyectado_roe,valor_proyectado_solvencia_proxy,puntaje_activos_proyectados,puntaje_roe_proyectado,puntaje_morosidad_proyectada,puntaje_solvencia_proyectada,puntaje_tendencia_general,score_salud,tipo_score,lectura_score,ranking_score,analisis_ejecutivo
0,2009,COFIEC,12,12.694167,8.422500,16.884167,13.357500,49.340000,1,-2.285805,1.028695,Pequeños con mayor solvencia proxy,75,16.884167,12.829900,4.054267,15.330507,96.15,32.932970,53.574122,28.992740,-13.772734,NaN,NaN,NaN,-47.574211,0.00,0.00,0.00,28.95,32.932970,53.574122,28.992740,15.177266,NaN,NaN,NaN,NaN,NaN,86.75,historico_sin_proyecciones,Salud financiera favorable,1,"En 2009, COFIEC obtiene un score de salud fina..."
1,2009,SUDAMERICANO,12,8.455000,5.943333,28.581667,9.866667,46.414167,1,-2.074907,0.192645,Pequeños con mayor solvencia proxy,75,28.581667,25.379492,3.202175,12.212957,92.31,2.087329,-7.988043,-9.039341,-5.220986,14.658207,-251.988746,-141.460729,-12.178647,14.24,3.17,6.39,42.87,16.327329,-4.818043,-2.649341,37.649014,NaN,NaN,NaN,NaN,NaN,84.62,historico_sin_proyecciones,Salud financiera favorable,2,"En 2009, SUDAMERICANO obtiene un score de salu..."
2,2009,DELBANK,12,16.391667,7.345833,20.746667,4.370833,34.221667,1,-1.542175,-0.128875,Pequeños con mayor solvencia proxy,75,20.746667,17.879211,2.867456,12.904694,88.46,3.525327,1.504959,-3.134782,-1.956932,9.301653,52.805591,-23.134919,-6.861612,37.90,2.85,13.55,28.52,41.425327,4.354959,10.415218,26.563068,NaN,NaN,NaN,NaN,NaN,82.48,historico_sin_proyecciones,Salud financiera favorable,3,"En 2009, DELBANK obtiene un score de salud fin..."
3,2009,PICHINCHA,12,4454.495833,455.401667,12.363333,3.361667,8.815000,0,1.491146,-0.105078,Bancos grandes o consolidados,85,12.363333,11.736673,0.626661,0.731940,73.08,2255.081828,-0.203012,-1.488791,-0.547767,10.092597,-4.822143,-12.979868,-5.921805,22343.92,4.21,11.47,9.25,24599.001828,4.006988,9.981209,8.702233,NaN,NaN,NaN,NaN,NaN,78.38,historico_sin_proyecciones,Salud financiera favorable,4,"En 2009, PICHINCHA obtiene un score de salud f..."
4,2009,TERRITORIAL,12,121.978333,7.700000,44.045833,9.289167,4.770833,2,-0.251752,-1.972609,Mayor riesgo relativo,30,44.045833,37.866093,6.179741,11.860391,100.00,0.990574,-1.613427,22.595684,2.814019,0.581869,NaN,NaN,39.029391,170.24,0.00,0.00,7.21,171.230574,-1.613427,22.595684,10.024019,NaN,NaN,NaN,NaN,NaN,68.89,historico_sin_proyecciones,Perfil financiero sólido,5,"En 2009, TERRITORIAL obtiene un score de salud..."
5,2009,FINCA,12,37.073333,10.245000,4.708333,12.186667,24.007500,1,-1.279572,0.692629,Pequeños con mayor solvencia proxy,75,4.708333,5.034159,-0.325825,0.843741,50.00,6.491991,-3.491624,57.422090,-0.377638,7.878630,-18.214001,67.619041,-4.051905,82.40,19.17,-84.92,9.32,88.891991,15.678376,-27.497910,8.942362,NaN,NaN,NaN,NaN,NaN,61.11,historico_sin_proyecciones,Perfil financiero sólido,6,"En 2009, FINCA obtiene un score de salud finan..."
6,2009,SOLIDARIO,12,294.878333,37.487500,10.470833,8.236667,9.635833,2,-0.494737,-0.238517,Mayor riesgo relativo,30,10.470833,8.497699,1.973134,7.938995,84.62,108.091598,0.277107,12.336648,-2.261535,13.795992,5.498163,127.050956,-12.571066,783.50,5.04,9.71,17.99,891.591598,5.317107,22.046648,15.728465,NaN,NaN,NaN,NaN,NaN,60.34,historico_sin_proyecciones,Perfil financiero sólido,7,"En 2009, SOLIDARIO obtiene un score de salud f..."
7,2009,AUSTRO,12,647.960833,57.102500,20.64666

## 10. Exportar score

In [11]:
columnas = [
    "anio", "tipo_score", "ranking_score", "banco_estandarizado", "score_salud", "lectura_score",
    "perfil_cluster", "puntaje_roe_modelo", "puntaje_cluster", "puntaje_tendencia_general",
    "puntaje_morosidad_proyectada", "puntaje_solvencia_proyectada", "roe_real", "roe_estimado",
    "error_roe", "activos_totales", "patrimonio", "roe", "morosidad", "solvencia_proxy",
    "cambio_proyectado_pct_activos_totales", "cambio_proyectado_roe", "cambio_proyectado_morosidad",
    "cambio_proyectado_solvencia_proxy", "analisis_ejecutivo"
]

columnas = [c for c in columnas if c in df_score.columns]

df_score_final = df_score[columnas].copy()
df_score_actual = df_score_final[df_score_final["anio"] == ANIO_SCORE_ACTUAL].copy()

exportar_csv_parquet(df_score_final, "score_salud_bancaria")
exportar_csv_parquet(df_score_actual, "score_salud_bancaria_actual")
exportar_csv_parquet(df_score, "score_componentes_bancos")

print("Score histórico:", df_score_final.shape)
print("Score actual:", df_score_actual.shape)

Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\score_salud_bancaria.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\score_salud_bancaria.parquet
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\score_salud_bancaria_actual.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\score_salud_bancaria_actual.parquet
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\score_componentes_bancos.csv
Exportado: C:\Users\eddy.trejo\Documents\Seminario_Titulacion\inteligencia-financiera-banca-privada-ecuador\01_datos_procesados\modelos\score_componentes_bancos.parquet
